In [21]:
import json
import csv
import math
import time
import warnings
from pathlib import Path
from langchain_chroma import Chroma

DOCS_DIR        = "../../data/parsed_json/"
CHROMA_BASE     = "../../data/vector_db_toc_overlap_subsplit"
GOLDEN_SET_PATH = "../../eval/golden_dataset/golden_dataset_v2.csv"
COLLECTION_NAME = "rfp_docs"
EMBEDDING_MODEL = "bge-m3"
BATCH_SIZE      = 500
TOP_K           = 10
CHUNK_SIZE      = 500
OVERLAP         = 100
MERGE_TABLE     = True

In [2]:
STRATEGIES = [
    {"name": "A_baseline",  "toc": False},  # 기존 방식: [섹션: A > B]
    {"name": "B_toc",       "toc": True},   # TOC 구조 활용: [대분류] [소분류]
]

GS_TO_DOCID = {
    "GKL_그룹웨어":            "D093",
    "KUSF_체육":               "D011",
    "강릉어선안전":             "D024",
    "경기_사회서비스":          "D087",
    "고려대학교_차세대포털":    "D008",
    "광주과기원_RCMS":          "D073",
    "광주과학기술원_학사시스템": "D039",
    "구미_육상":               "D018",
    "국립중앙의료원_응급":      "D069",
    "국민연금공단_이러닝":      "D049",
    "국민연금_멀티턴1":         "D050",
    "국민연금_멀티턴2":         "D050",
    "국민연금_멀티턴3":         "D050",
    "국방_대용량":             "D010",
    "기초과학연구원_극저온":    "D051",
    "나노종합_팹":             "D099",
    "대검찰청_홈페이지":        "D053",
    "민속박물관_아카이브":      "D090",
    "벤처협회_시스템":          "D086",
    "보험개발원_실손":          "D083",
    "봉화군_재난":             "D005",
    "부산관광_ERP":            "D037",
    "서민금융_채팅":            "D056",
    "서영대_교육":             "D045",
    "서울_디지털성범죄":        "D068",
    "서울_지도플랫폼":          "D040",
    "서울교육청_ISP":           "D084",
    "세종_인사":               "D088",
    "우즈벡_관개":             "D072",
    "울산_버스":               "D034",
    "인천_도시계획":            "D004",
    "인천_일자리":             "D030",
    "인천공항_ERP":            "D079",
    "적십자_재해복구":          "D095",
    "철도_ISP":               "D070",
    "통합정보시스템_충돌":      "D016",
    "평택_버스":               "D060",
    "해양박물관_자료":          "D066",
    # 다중 문서
    "고려대_vs_광주과기원":     ["D008", "D039"],
    "버스_다중비교":            ["D034", "D060"],
    "재난안전_종합":            ["D005", "D007"],
    "철도_vs_인천_ISP":        ["D070", "D030"],
    # 평가 제외
    "TEST": None, "unknown": None, "ISP_다중비교": None,
    "교육관련_다중문서": None, "문화_다중비교": None,
    "의료_다중문서": None, "재공고_종합": None,
    "신규_vs_고도화": None, "예산_최소_최대": None,
    "멀티턴_심화1": None, "멀티턴_심화2": None,
    "모른다_테스트1": None, "모른다_테스트2": None,
    "모른다_테스트3": None, "모른다_테스트4": None,
    "모른다_테스트5": None, "모른다_테스트6": None,
    "존재하지않는사업": None, "입찰마감_확인": None,
}

In [3]:
def clean(val):
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [4]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }


In [5]:
def build_toc_lookup(doc: dict) -> tuple[dict, dict]:
    by_order = {}
    by_title = {}
    for item in doc.get("toc", []):
        by_order[item["order"]] = item
        by_title[item.get("title", "")] = item
    return by_order, by_title

In [6]:
def build_section_header(
    header_path: list,
    toc_order: int | None,
    toc_by_order: dict,
    toc_by_title: dict,
    use_toc: bool,
) -> str:
    if not use_toc:
        return f"[섹션: {' > '.join(header_path)}]"

    level_labels = {1: "대분류", 2: "소분류", 3: "세부항목"}
    parts = []
    for i, title in enumerate(header_path):
        is_leaf = (i == len(header_path) - 1)
        # leaf는 toc_ref로 정확하게, 상위는 title 매핑 시도
        info = toc_by_order.get(toc_order) if is_leaf else toc_by_title.get(title)
        if info:
            label = level_labels.get(info["level"], "항목")
            parts.append(f"[{label}] {info['number']}. {title}")
        else:
            parts.append(f"[항목] {title}")
    return "\n".join(parts)

In [7]:
def get_toc_ref_order(toc_ref) -> int | None:
    if isinstance(toc_ref, int):
        return toc_ref
    if isinstance(toc_ref, dict):
        return toc_ref.get("order")
    return None

In [20]:
def chunk_section(section: dict, toc_by_order: dict, toc_by_title: dict, use_toc: bool) -> list[dict]:
    header_path = section.get("header_path", [])
    toc_ref = section.get("toc_ref")
    order = get_toc_ref_order(toc_ref)
    section_header = build_section_header(header_path, order, toc_by_order, toc_by_title, use_toc)

    results = []
    current_text = ""
    last_text_block = None

    def flush(block):
        nonlocal current_text, last_text_block
        if current_text.strip():
            results.append({"content": f"{section_header}\n\n{current_text.strip()}", "block": block})
            current_text = current_text[-OVERLAP:] if OVERLAP else ""
            last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue

        content = block["content"]

        # needs_subsplit: 블록 내부를 overlap 포함해서 분할
        if block.get("needs_subsplit") and len(content) > CHUNK_SIZE:
            flush(last_text_block)
            i = 0
            while i < len(content):
                sub = content[i:i+CHUNK_SIZE]
                next_i = i + CHUNK_SIZE - OVERLAP
                is_last = next_i >= len(content)
                if is_last and len(sub) < OVERLAP * 2:
                    # 너무 작은 마지막 조각 → current_text에 넣어 다음 블록과 합치기
                    current_text = sub + "\n\n"
                    last_text_block = block
                else:
                    results.append({"content": f"{section_header}\n\n{sub}", "block": block})
                i = next_i
            continue

        if block["type"] == "table":
            if MERGE_TABLE:
                combined = (current_text + "\n\n" + content).strip()
                if len(combined) > CHUNK_SIZE and current_text.strip():
                    flush(last_text_block)
                    results.append({"content": f"{section_header}\n\n{content}", "block": block})
                else:
                    current_text = combined + "\n\n"
                    last_text_block = block
            else:
                flush(last_text_block)
                results.append({"content": f"{section_header}\n\n{content}", "block": block})
        else:
            if len(current_text) + len(content) > CHUNK_SIZE and current_text.strip():
                flush(last_text_block)
                current_text = current_text + content + "\n\n"
            else:
                current_text += content + "\n\n"
            last_text_block = block

    if current_text.strip():
        results.append({"content": f"{section_header}\n\n{current_text.strip()}", "block": last_text_block})

    return results

In [9]:
def process_doc(doc: dict, use_toc: bool) -> tuple[list[str], list[dict]]:
    texts, metas = [], []
    meta = doc.get("metadata", {})
    name = str(clean(meta.get("사업명")))
    org  = str(clean(meta.get("발주기관")))
    prefix = f"[사업명] {name}\n[발주기관] {org}\n\n"

    if use_toc:
        toc_by_order, toc_by_title = build_toc_lookup(doc)
    else:
        toc_by_order, toc_by_title = {}, {}

    for section in doc.get("sections", []):
        chunks = chunk_section(section, toc_by_order, toc_by_title, use_toc)
        for item in chunks:
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))

    return texts, metas

In [10]:
def load_embedding_model(name: str):
    if name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "text-embedding-3-small":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")

In [11]:
def save_vectorstore(strategy: dict, all_texts: list, all_metas: list, embedding_model):
    chroma_dir = f"{CHROMA_BASE}/{strategy['name']}"

    if Path(chroma_dir).exists():
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        existing_count = vs._collection.count()
        if existing_count >= len(all_texts):
            print(f"[SKIP] {strategy['name']} — 완료 ({existing_count}개 청크)")
            return vs
        print(f"[RESUME] {strategy['name']} — {existing_count}/{len(all_texts)}개, 이어서 진행")
        start_from = existing_count
    else:
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        start_from = 0

    for start in range(start_from, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vs.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"  완료 ({vs._collection.count()}개 청크) → {chroma_dir}")
    return vs

In [12]:
def hit(retrieved_ids, target_ids, k):
    return any(tid in retrieved_ids[:k] for tid in target_ids)

In [ ]:
# 골든셋 로드
golden_set = []
with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        golden_set.append(row)

golden_set = golden_set[:101]
print(f"골든셋 문항 수: {len(golden_set)}")

json_files = sorted(Path(DOCS_DIR).glob("*.json"))
print(f"JSON 파일 수: {len(json_files)}")

embedding_model = load_embedding_model(EMBEDDING_MODEL)
warnings.filterwarnings("ignore", message="Relevance scores must be between")

In [14]:
# 전략별 실험 
eval_results = []

for strategy in STRATEGIES:
    print(f"\n{'='*50}")
    print(f"[전략] {strategy['name']} (toc={strategy['toc']})")

    all_texts, all_metas = [], []
    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        texts, metas = process_doc(doc, use_toc=strategy["toc"])
        all_texts.extend(texts)
        all_metas.extend(metas)

    print(f"  총 청크: {len(all_texts)}개 | 평균 길이: {sum(len(t) for t in all_texts)//len(all_texts)}자")

    vs = save_vectorstore(strategy, all_texts, all_metas, embedding_model)

    recall3, recall5, recall10, mrr_scores, query_times = [], [], [], [], []
    skipped = 0

    for row in golden_set:
        gs_key = str(row["doc_id"])
        target = GS_TO_DOCID.get(gs_key)

        if target is None:
            skipped += 1
            continue

        target_ids = target if isinstance(target, list) else [target]

        t0 = time.time()
        hits = vs.similarity_search_with_relevance_scores(row["question"], k=TOP_K)
        query_times.append(time.time() - t0)

        retrieved_doc_ids = [doc.metadata.get("doc_id", "") for doc, _ in hits]

        recall3.append(1.0 if hit(retrieved_doc_ids, target_ids, 3) else 0.0)
        recall5.append(1.0 if hit(retrieved_doc_ids, target_ids, 5) else 0.0)
        recall10.append(1.0 if hit(retrieved_doc_ids, target_ids, 10) else 0.0)

        rank = next(
            (i + 1 for i, d in enumerate(retrieved_doc_ids) if d in target_ids),
            None,
        )
        mrr_scores.append(1.0 / rank if rank else 0.0)

    n = len(recall3)
    print(f"  평가: {n}개 | 제외: {skipped}개")

    eval_results.append({
        "전략":      strategy["name"],
        "toc":       strategy["toc"],
        "Recall@3":  round(sum(recall3)    / n, 4),
        "Recall@5":  round(sum(recall5)    / n, 4),
        "Recall@10": round(sum(recall10)   / n, 4),
        "MRR":       round(sum(mrr_scores) / n, 4),
        "avg_ms":    round(sum(query_times) / n * 1000, 1),
    })




[전략] A_baseline (toc=False)
  총 청크: 17889개 | 평균 길이: 491자
  저장 완료: 500/17889
  저장 완료: 1000/17889
  저장 완료: 1500/17889
  저장 완료: 2000/17889
  저장 완료: 2500/17889
  저장 완료: 3000/17889
  저장 완료: 3500/17889
  저장 완료: 4000/17889
  저장 완료: 4500/17889
  저장 완료: 5000/17889
  저장 완료: 5500/17889
  저장 완료: 6000/17889
  저장 완료: 6500/17889
  저장 완료: 7000/17889
  저장 완료: 7500/17889
  저장 완료: 8000/17889
  저장 완료: 8500/17889
  저장 완료: 9000/17889
  저장 완료: 9500/17889
  저장 완료: 10000/17889
  저장 완료: 10500/17889
  저장 완료: 11000/17889
  저장 완료: 11500/17889
  저장 완료: 12000/17889
  저장 완료: 12500/17889
  저장 완료: 13000/17889
  저장 완료: 13500/17889
  저장 완료: 14000/17889
  저장 완료: 14500/17889
  저장 완료: 15000/17889
  저장 완료: 15500/17889
  저장 완료: 16000/17889
  저장 완료: 16500/17889
  저장 완료: 17000/17889
  저장 완료: 17500/17889
  저장 완료: 17889/17889
  완료 (17889개 청크) → ../../data/vector_db_toc_overlap_subsplit/A_baseline
  평가: 84개 | 제외: 17개

[전략] B_toc (toc=True)
  총 청크: 17889개 | 평균 길이: 497자
  저장 완료: 500/17889
  저장 완료: 1000/17889
  저장 완료: 1500/17889
  저

In [15]:
# 결과 출력 
from IPython.display import Markdown, display

header = "| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |\n|------|----------|----------|-----------|-----|--------|"
rows = "\n".join(
    f"| {r['전략']} | {r['Recall@3']} | {r['Recall@5']} | {r['Recall@10']} | {r['MRR']} | {r['avg_ms']} |"
    for r in eval_results
)
display(Markdown(f"## TOC 구조 기반 청킹 실험 결과\n\n{header}\n{rows}"))

## TOC 구조 기반 청킹 실험 결과

| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |
|------|----------|----------|-----------|-----|--------|
| A_baseline | 0.2262 | 0.2381 | 0.2976 | 0.2171 | 25.5 |
| B_toc | 0.25 | 0.2619 | 0.3095 | 0.2172 | 24.9 |

In [16]:
total_sections = 0
matched_sections = 0

for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        doc = json.load(f)
    
    toc_by_order = {item["order"]: item for item in doc.get("toc", [])}
    
    for section in doc.get("sections", []):
        total_sections += 1
        order = get_toc_ref_order(section.get("toc_ref"))
        if order is not None and order in toc_by_order:
            matched_sections += 1

print(f"매핑 성공률: {matched_sections}/{total_sections} ({matched_sections/total_sections*100:.1f}%)")

매핑 성공률: 2936/6515 (45.1%)


In [17]:
from collections import Counter
toc_ref_types = Counter()
for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        doc = json.load(f)
    for section in doc.get("sections", []):
        toc_ref_types[type(section.get("toc_ref")).__name__] += 1
print(toc_ref_types)

Counter({'NoneType': 3579, 'dict': 2899, 'int': 37})


In [18]:
# 미매핑 섹션 샘플 확인
miss_samples = []

for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        doc = json.load(f)
    toc_by_order = {item["order"]: item for item in doc.get("toc", [])}
    
    for section in doc.get("sections", []):
        order = get_toc_ref_order(section.get("toc_ref"))
        if order is None:
            miss_samples.append({"reason": "toc_ref없음", "header_path": section.get("header_path")})
        elif order not in toc_by_order:
            miss_samples.append({"reason": f"order={order} toc에없음", "header_path": section.get("header_path"), "toc_ref": section.get("toc_ref")})
        
        if len(miss_samples) >= 10:
            break
    if len(miss_samples) >= 10:
        break

for s in miss_samples:
    print(s)

{'reason': 'toc_ref없음', 'header_path': ['(서두)']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ 사업금액 : 129,300,000원(부가세 포함)']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ UICC 시스템 기능개선 ※ 세부 내용은 “Ⅳ. 제안요청사항” 참고']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ 관리자 활용도 제고를 위한 기능 개선']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ 실태조사 항목 지침 변경사항을 반영하고 시스템 기능을 개선하여 사용자 편의성 도모']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ 능력과 경험이 풍부한 전문사업자를 주관사업자로 선정']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ 정보기술 활용전략']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ 보안 전략']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '3. 추진체계']}
{'reason': 'toc_ref없음', 'header_path': ['Ⅵ. 제안서 작성요령75', '□ 추진일정표']}
